In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

try:
    from tqdm import tqdm
except ImportError:
    !pip install tqdm
    from tqdm import tqdm

# 1. Direct DFT computation O(N^2)
def slow_dft(x):
    N = len(x)
    X = np.zeros(N, dtype=complex)
    for k in range(N):
        for n in range(N):
            angle = -2 * np.pi * k * n / N
            X[k] += x[n] * complex(np.cos(angle), np.sin(angle))
    return X

# 2. Fast Fourier Transform (FFT) Radix-2 DIT O(N log2 N)
def fast_fft(x):
    N = len(x)
    if (N & (N - 1)) != 0 or N == 0:
        raise ValueError("N must be a power of 2.")
    
    X = [complex(val, 0) for val in x]
    
    # Bit-reversal sorting
    bits = int(np.log2(N))
    for i in range(N):
        rev = 0
        for j in range(bits):
            if (i & (1 << j)) != 0:
                rev |= (1 << (bits - 1 - j))
        if i < rev:
            X[i], X[rev] = X[rev], X[i]
            
    # Butterfly structures
    length = 2
    while length <= N:
        half_len = length // 2
        angle = -2 * np.pi / length
        wlen = complex(np.cos(angle), np.sin(angle))
        
        for i in range(0, N, length):
            w = complex(1, 0)
            for j in range(half_len):
                u = X[i + j]
                v = X[i + j + half_len] * w
                X[i + j] = u + v
                X[i + j + half_len] = u - v
                w *= wlen
        length *= 2
    return X

# 3. Execution time benchmarking (Extended up to 4096 for better visualization)
sizes = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
dft_times = []
fft_times = []

print("\033[1mBenchmarking Computational Complexity: Direct DFT vs. FFT\033[0m")
print("Please wait while computations are being performed...")

for N in tqdm(sizes, desc="Benchmarking Progress", ncols=80):
    x = np.random.rand(N)
    
    # Measure DFT execution time (Skipping very large N for slow_dft to avoid excessive waiting)
    if N <= 2048:
        start = time.perf_counter()
        slow_dft(x)
        dft_times.append((time.perf_counter() - start) * 1000)
    else:
        dft_times.append(np.nan) # Estimated/Skipped for extremely slow values
        
    # Measure FFT execution time
    start = time.perf_counter()
    fast_fft(x)
    fft_times.append((time.perf_counter() - start) * 1000)

# 4. Plotting with Logarithmic Scale on Y-axis
plt.figure(figsize=(10, 5))
plt.plot(sizes[:len(dft_times)], dft_times, 'o-', color='crimson', label='Direct DFT $O(N^2)$', linewidth=2)
plt.plot(sizes, fft_times, 's-', color='dodgerblue', label='FFT Algorithm $O(N \\log_2 N)$', linewidth=2)

# Εφαρμογή λογαριθμικής κλίμακας στον άξονα Y
plt.yscale('log')

plt.xlabel('Sample Size ($N$)', fontsize=12)
plt.ylabel('Execution Time (ms) - Log Scale', fontsize=12)
plt.title('Computational Complexity Comparison: DFT vs FFT (Logarithmic Scale)', fontsize=14, fontweight='bold')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=11)
plt.grid(True, which="both", linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()